# Summer Analytics Week 2 Hackathon
## Starter Notebook

This notebook walks you through a simple end-to-end pipeline to get your first submission on the leaderboard. It covers:

- Loading the provided datasets
- Basic preprocessing (handling missing values & encoding)
- Training a **Logistic Regression** baseline model
- Generating predictions for `private_test.csv`
- Creating a valid `submission.csv`

> **Note:** This is just a starting point. Feel free to experiment and improve upon this baseline!


### Step 1: Import Libraries

We start by importing `pandas` for loading and manipulating data, and a few utilities from `scikit-learn` for preprocessing and modelling.
These are all standard libraries that come pre-installed in most Python environments like Kaggle, Colab, or Anaconda.
No additional installations are required to run this notebook.


In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


### Step 2: Load Data

We read all three CSV files — `train.csv`, `public_test.csv`, and `private_test.csv` — into pandas DataFrames.
The training set contains labelled examples we will learn from, while the private test set is what we need to make final predictions on.
Printing the shapes gives a quick sanity check that the files loaded correctly.


In [2]:
train = pd.read_csv("./dataset/train.csv")
public_test = pd.read_csv("./dataset/public_test.csv")
private_test = pd.read_csv("./dataset/private_test.csv")

print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test.shape)

train.head()


Train Shape: (10000, 14)
Public Test Shape: (3000, 14)
Private Test Shape: (3000, 13)


,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


### Step 3: Define Features and Target

We separate the target column (`Converted`) from the rest of the features in the training set.
For simplicity, we only keep **numeric columns** in this baseline — categorical columns are dropped for now.
This keeps things easy to understand before we add more advanced preprocessing.


In [3]:
TARGET = "Converted"

# Separate features and target
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]

# Keep only numeric columns for this baseline
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
X_train = X_train[numeric_cols]
X_private = private_test[numeric_cols]

print("Features used:", numeric_cols)


Features used: ['User_ID', 'Age', 'Income', 'City_Tier', 'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases', 'Discount_Seen', 'Browser_Version', 'Campaign_Code']


### Step 4: Handle Missing Values & Scale Features

Real-world datasets often have missing values — we fill them in using the **median** of each column, which is robust to outliers.
After that, we apply **Standard Scaling** to bring all features onto the same scale, which is important for Logistic Regression to work well.
We fit both the imputer and scaler only on the training data, then apply the same transformation to the test data to avoid data leakage.


In [4]:
# Fill missing values with the median of each column
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_private_imputed = imputer.transform(X_private)

# Scale features to zero mean and unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_private_scaled = scaler.transform(X_private_imputed)

print("Preprocessing done! Training data shape:", X_train_scaled.shape)


Preprocessing done! Training data shape: (10000, 11)


### Step 5: Train a Logistic Regression Model

Logistic Regression is one of the simplest and most interpretable classification algorithms — a great starting point for any binary classification problem.
It models the probability that a given input belongs to a class using a linear combination of features passed through a sigmoid function.
We set `max_iter=1000` to ensure the solver has enough iterations to converge on this dataset.


In [5]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print("Training Accuracy: {:.4f}".format(model.score(X_train_scaled, y_train)))


Model trained successfully!
Training Accuracy: 0.7097


### Step 6: Generate Predictions and Create Submission

We run the trained model on the scaled private test data to generate our final predictions.
The predictions are combined with the `User_ID` column into a DataFrame matching the required submission format.
The file is saved as `submission.csv` — just upload this to the competition portal to appear on the leaderboard!


In [6]:
predictions = model.predict(X_private_scaled)

submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
submission.head()


submission.csv created successfully!


,User_ID,Converted
0,103001,0
1,103002,0
2,103003,0
3,103004,0
4,103005,0


In [8]:
from sklearn.metrics import f1_score

# 1. Replicate their exact numeric filtering for the public test set
X_public = public_test[numeric_cols]
y_public_actual = public_test["Converted"]

# 2. Transform the public test data using their existing baseline imputer and scaler
X_public_imputed = imputer.transform(X_public)
X_public_scaled = scaler.transform(X_public_imputed)

# 3. Predict using their trained model and calculate the baseline F1-score
baseline_preds = model.predict(X_public_scaled)
baseline_f1 = f1_score(y_public_actual, baseline_preds)

print(f"Instructors' Baseline Public Test F1-Score: {baseline_f1:.4f}")

Instructors' Baseline Public Test F1-Score: 0.3550
